# Houston and Los Angeles outdoor heat baselines
Train on 2018–2024 and validate on 2025. Each model estimates a typical outdoor feels-like high for its city; the report also displays the independent live weather forecast.


In [ ]:
from datetime import date
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import urlopen
import json
import math
import numpy as np

def fit_city(city, lat, lon, zone):
    query = urlencode({
        "latitude": lat, "longitude": lon,
        "start_date": "2018-01-01", "end_date": "2025-12-31",
        "daily": "apparent_temperature_max", "temperature_unit": "fahrenheit",
        "timezone": zone, "models": "era5_seamless",
    })
    with urlopen("https://archive-api.open-meteo.com/v1/archive?" + query, timeout=45) as response:
        daily = json.load(response)["daily"]

    def features(day):
        a = (day - date(day.year, 1, 1)).days * 2 * math.pi / 365.25
        return [1, math.sin(a), math.cos(a), math.sin(2*a), math.cos(2*a)]

    samples = [(date.fromisoformat(day), value) for day, value in zip(daily["time"], daily["apparent_temperature_max"]) if value is not None]
    train = [(features(day), value) for day, value in samples if day.year < 2025]
    test = [(features(day), value) for day, value in samples if day.year == 2025]
    assert len(train) > 2000 and len(test) >= 350, "Historical samples are incomplete"
    x, y = zip(*train)
    coefficients = np.linalg.lstsq(x, y, rcond=None)[0]
    test_x, test_y = zip(*test)
    prediction = np.sum(np.asarray(test_x) * coefficients, axis=1)
    mae = np.mean(np.abs(prediction - np.asarray(test_y)))
    artifact = {
        "city": city, "trainedThrough": "2024-12-31",
        "holdout": "2025", "holdoutMae": round(float(mae), 3),
        "samples": len(train), "coefficients": [round(float(c), 7) for c in coefficients],
    }
    return artifact

models = [
    fit_city("Houston", 29.685, -95.400, "America/Chicago"),
    fit_city("Los Angeles", 34.052, -118.246, "America/Los_Angeles"),
]
destination = Path("backend/config/heat-models.json") if Path("backend/config/property.demo.json").exists() else Path("heat-models.json")
destination.write_text(json.dumps(models, indent=2) + "\n")
for model in models:
    print(json.dumps(model))
